# Motor Exercise 2 — plotting encoder counts and wheel speed

Load timestamped encoder snapshots, inspect the recorded counts, and calculate left and right wheel speed using the measured time interval.

Start with the supplied synthetic example so that every cell runs before you have collected data. The example demonstrates the plotting route; it is not evidence about your robot and is not a result you should expect to reproduce. When you are ready, change only the settings in **Use the example or your own data** and run the notebook again.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
rng = np.random.default_rng(2026)


## 1. Use the example or your own data

Leave `USE_EXAMPLE_DATA` set to `True` on your first run. To use your measurements, upload the CSV, set it to `False`, and enter the filename. This is the main cell you need to edit.

Expected CSV columns: `trial_num`, `sample_num`, `timestamp_ms`, `requested_left_pwm`, `requested_right_pwm`, `left_encoder_count`, and `right_encoder_count`.

Requested PWM is a command. Encoder-count change is the recorded wheel motion used to estimate speed.


In [ ]:
USE_EXAMPLE_DATA = True
CSV_FILENAME = "motor_exercise02_encoder_snapshots.csv"

print("Using:", "synthetic example" if USE_EXAMPLE_DATA else CSV_FILENAME)


## 2. Create the small synthetic example

You do not need to understand or edit this generation code. It creates one trial with slightly irregular sampling intervals.


In [ ]:
sample_count = 70
interval_ms = rng.integers(48, 54, sample_count)
timestamp_ms = np.cumsum(interval_ms)
left_rate_cps = np.where(timestamp_ms < 800, 0, 610)
right_rate_cps = np.where(timestamp_ms < 800, 0, 575)

example_data = pd.DataFrame({
    "trial_num": 1,
    "sample_num": np.arange(sample_count),
    "timestamp_ms": timestamp_ms,
    "requested_left_pwm": np.where(timestamp_ms < 800, 0, 80),
    "requested_right_pwm": np.where(timestamp_ms < 800, 0, 80),
    "left_encoder_count": np.rint(
        np.cumsum(left_rate_cps * interval_ms / 1000)
    ).astype(int),
    "right_encoder_count": np.rint(
        np.cumsum(right_rate_cps * interval_ms / 1000)
    ).astype(int),
})


## 3. Load and preview the selected data

This is where your uploaded CSV enters the notebook. Check the first rows before continuing: column names, units and labels should match the exercise.


In [ ]:
if USE_EXAMPLE_DATA:
    data = example_data.copy()
else:
    data = pd.read_csv(CSV_FILENAME)

data.head()


## 4. Plot the recorded encoder counts

`melt(...)` places the left and right count columns into one plotting column. The original table remains unchanged.


In [ ]:
count_plot_data = data.melt(
    id_vars=["trial_num", "timestamp_ms"],
    value_vars=["left_encoder_count", "right_encoder_count"],
    var_name="wheel",
    value_name="encoder_count",
)

sns.lineplot(
    data=count_plot_data,
    x="timestamp_ms",
    y="encoder_count",
    hue="wheel",
    estimator=None,
)
plt.title("Recorded encoder counts")
plt.xlabel("Timestamp (ms)")
plt.ylabel("Encoder count")
plt.show()


## 5. Calculate speed from consecutive snapshots

Within each trial, `diff()` subtracts consecutive timestamps and counts. Dividing count change by elapsed seconds gives encoder counts per second.


In [ ]:
data = data.sort_values(["trial_num", "sample_num"]).copy()
data["elapsed_s"] = data.groupby("trial_num")["timestamp_ms"].diff() / 1000

data["left_speed_cps"] = (
    data.groupby("trial_num")["left_encoder_count"].diff()
    / data["elapsed_s"]
)
data["right_speed_cps"] = (
    data.groupby("trial_num")["right_encoder_count"].diff()
    / data["elapsed_s"]
)

data[["timestamp_ms", "elapsed_s", "left_speed_cps", "right_speed_cps"]].head(8)


## 6. Plot the derived wheel-speed estimates


In [ ]:
speed_plot_data = data.melt(
    id_vars=["trial_num", "timestamp_ms"],
    value_vars=["left_speed_cps", "right_speed_cps"],
    var_name="wheel",
    value_name="speed_cps",
).dropna()

sns.lineplot(
    data=speed_plot_data,
    x="timestamp_ms",
    y="speed_cps",
    hue="wheel",
    estimator=None,
)
plt.axhline(0, color="black", linewidth=1)
plt.title("Wheel speed calculated from encoder snapshots")
plt.xlabel("Timestamp (ms)")
plt.ylabel("Encoder speed (counts/s)")
plt.show()


## What to notice

- Do the encoder counts change in the direction you observed physically?
- How much does the calculated speed vary when the wheel appears steady?
- Do the left and right wheels report similar speeds for the same command?
- The speed trace is derived from encoder motion; the PWM trace is only the request.
